## GTEx colocalization

In [2]:
set.seed(2026)
GTEx_genes <- read.table("/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/gtex/GTEx_genes_ensg.txt",
                         header = TRUE)
sampled <- GTEx_genes[sample(nrow(GTEx_genes), 5000), ,drop = FALSE]
write.table(sampled,
            "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/gtex/GTEx_genes_5000.txt",
            quote = FALSE, row.names = FALSE)

In [1]:
%%writefile /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/code/Gtex_fine.R
# ============================================================
# GTEx eQTL fine-mapping with LD sketch reference
#   - SuSiE (individual)
#   - SuSiE-RSS with sketch (no EB)
#   - SuSiE-RSS with sketch + R_finite EB correction
#
# ============================================================

# ---- batch index from command line ----
# ---- batch index from command line ----
args <- commandArgs(trailingOnly = TRUE)
IDX  <- as.integer(args[1])
stopifnot(!is.na(IDX), IDX >= 1)

# ---- libraries ----
library(susieR)
library(data.table)
library(tidyverse)
library(BEDMatrix)
library(vroom)
library(mixsqp)
library(pecotmr)
library(pgenlibr)
library(matrixStats)
library(crayon)
library(Rfast)

# ---- local source overrides ----
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)

# ============================================================
# Two susieR builds, each ISOLATED in its own environment so their
# identically-named functions never clobber.
#   susie_hc / susie_indiv : haochen build (susie_rss + individual-data susie)
#   susie_xw               : xuewei build  (R_mismatch = "eb_mix")
# ============================================================
source_build <- function(dir) {
  rdir <- if (dir.exists(file.path(dir, "R"))) file.path(dir, "R") else dir
  env  <- new.env(parent = globalenv())
  for (f in list.files(rdir, full.names = TRUE, pattern = "\\.R$"))
    sys.source(f, envir = env)
  env
}

susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss
susie_xw <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/susieR_July17_2026_singleL/")$susie_rss
susie_indiv  <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie
# ---- helpers ----
safe_susie <- function(.fn, .label, ...) {
  tryCatch(
    withCallingHandlers(
      .fn(...),
      warning = function(w) invokeRestart("muffleWarning")
    ),
    error = function(e) {
      cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
      NULL
    }
  )
}

# max inter-CS correlation (NA when < 2 CS), against the LD the fit used
cs_maxcor <- function(model, Xcorr)
  tryCatch(as.numeric(get_cs_correlation(model, Xcorr = Xcorr, max = TRUE)),
           error = function(e) NA_real_)

sketch_harmonize <- function(target_data, ref_variants, sketch) {
  qc_data <- allele_qc(match_min_prop = 0,
                       target_data    = target_data,
                       ref_variants   = ref_variants)
  qced <- qc_data$target_data_qced
  idx  <- match(qced$variant_id, colnames(sketch))
  if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
  sketch2 <- sketch[, idx, drop = FALSE]
  flip_index <- which(qced$variants_id_original != qced$variant_id)
  if (length(flip_index) > 0)
    sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
  colnames(sketch2) <- qced$variants_id_original
  list(sumstat = qced, sketch = sketch2,
       flipped_prop = length(flip_index) / ncol(sketch2))
}

kriging_fix <- function(z, LD, n, z_thres = 2, logLR_thres = 2) {
  krig <- kriging_rss(z = z, R = LD, n = n)
  krig_outlier <- krig$conditional_dist %>%
    mutate(index = row_number()) %>%
    filter(abs(z) > z_thres & logLR > logLR_thres)
  z_fix <- z
  z_fix[krig_outlier$index] <- -z_fix[krig_outlier$index]
  list(z = z_fix,
       outlier_index  = krig_outlier$index,
       outlier_number = length(krig_outlier$index),
       output         = krig$conditional_dist)
}

# ---- config ----
geneTSS_dir  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/GTEx/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list"
genotype_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/GTEx/genotype"
eqtl_dir     <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/GTEx/normalized_expression"
cov_dir      <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/GTEx/GTEx_Analysis_v8_eQTL_covariates"
ld_meta_file <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/ld_meta_file.tsv"
gene_list    <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/gtex/GTEx_genes_5000.txt"
out_dir      <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/res"

maf.cut            <- 0.05
missing.cut        <- 0.1
tissues_considered <- NULL
use_data_table     <- FALSE

B_insample  <- 1e6      # in-sample LD is exact -> huge R_finite (EB captures only residual)
B_sketch    <- 10000    # finite-reference size for the KG / sketch panel

dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

GTEx_genes <- read.table(gene_list, header = TRUE)
idx_start  <- 50 * (IDX - 1) + 1
idx_end    <- min(50 * IDX, nrow(GTEx_genes))

message(sprintf("[IDX=%d] processing genes %d-%d of %d",
                IDX, idx_start, idx_end, nrow(GTEx_genes)))

# ============================================================
# Outer loop: genes
# ============================================================
geneTSS <- read_delim(geneTSS_dir, show_col_types = FALSE)

for (i in idx_start:idx_end) {

  gene.name <- GTEx_genes$x[i]

  # skip-existing
  if (length(grep(gene.name, list.files(out_dir))) != 0) next
  print(i)

  message(sprintf("[%s] === gene %d/%d: %s ===",
                  format(Sys.time(), "%Y-%m-%d %H:%M:%S"), i, idx_end, gene.name))

  # ---- 1. gene TSS / region ----
  gene.TSS <- geneTSS %>%
    filter(gene_id == gene.name) %>%
    mutate(CHR        = str_extract(`#chr`, "(?<=chr)\\d+"),
           gene_start = pmax(0, start - 1e6),
           gene_end   = start + 1e6) %>%
    slice(1)

  if (nrow(gene.TSS) == 0) {
    saveRDS("gene not found", file.path(out_dir, paste0("nores_", gene.name, ".rds")))
    next
  }

  chr        <- gene.TSS$CHR
  gene.start <- gene.TSS$gene_start
  gene.end   <- gene.TSS$gene_end

  # skip chrX
  if (is.na(chr)) {
    message(sprintf("  [%s] Skipping chrX", gene.name))
    next
  }

  # ---- 2. individual genotype ----
  message(sprintf("  [%s] Loading genotype (chr %s)", gene.name, chr))
  genotype <- BEDMatrix(file.path(genotype_dir, paste0("GTEx_v8_genotype.", chr)))

  bim_data <- {
    path <- file.path(genotype_dir, paste0("GTEx_v8_genotype.", chr, ".bim"))
    if (use_data_table) read_delim(path, col_names = FALSE, show_col_types = FALSE)
    else                read.delim(path, header = FALSE, stringsAsFactors = FALSE)
  }

  gene.snp <- bim_data %>%
    mutate(idx = row_number()) %>%
    filter(V4 >= gene.start & V4 <= gene.end)

  if (nrow(gene.snp) == 0) {
    saveRDS("no SNPs in region", file.path(out_dir, paste0("nores_", gene.name, ".rds")))
    next
  }

  geno_mat <- data.matrix(genotype[, gene.snp$idx])
  rownames(geno_mat) <- str_replace(rownames(geno_mat), "0_", "")
  colnames(geno_mat) <- paste0("chr", gene.snp$V1, ":", gene.snp$V4, ":",
                               gene.snp$V6, ":", gene.snp$V5)

  # ---- 3. LD sketch ----
  region <- sprintf("chr%s:%d-%d", chr, gene.snp$V4[1], gene.snp$V4[nrow(gene.snp)])
  message(sprintf("  [%s] Loading sketch for region %s", gene.name, region))

  res_sketch           <- load_LD_matrix(ld_meta_file, region, return_genotype = TRUE)
  res_sketch$LD_matrix <- scale(res_sketch$LD_matrix, center = TRUE, scale = TRUE)
  sketch_ref           <- res_sketch$ref_panel

  # ---- 4. tissue loop ----
  eqtl_files <- list.files(eqtl_dir, pattern = "\\.bed\\.gz$")
  if (!is.null(tissues_considered))
    eqtl_files <- eqtl_files[str_detect(eqtl_files, paste(tissues_considered, collapse = "|"))]

  res_susie <- list()

  for (tissue.file in eqtl_files) {

    tissue <- str_extract(tissue.file, "^[^.]+")
    message(sprintf("[%s]   [%s] tissue: %s",
                    format(Sys.time(), "%Y-%m-%d %H:%M:%S"), gene.name, tissue))

    # expression
    expr <- {
      path <- file.path(eqtl_dir, tissue.file)
      if (use_data_table) read_delim(path, show_col_types = FALSE)
      else                read.delim(path, stringsAsFactors = FALSE)
    }
    expr$gene_id <- sub("\\..*", "", expr$gene_id)
    expr <- filter(expr, gene_id == gene.name)
    if (nrow(expr) == 0) {
      message(sprintf("  [%s] No expression in %s - skipping", gene.name, tissue))
      next
    }

    samples <- colnames(expr)[-c(1:4)]
    if (!use_data_table) samples <- gsub("\\.", "-", samples)

    # residualize on covariates
    covar <- read_delim(file.path(cov_dir, paste0(tissue, ".v8.covariates.txt")),
                        show_col_types = FALSE) %>%
      select(-ID) %>% as.matrix()
    covar        <- covar[, samples]
    covar_design <- cbind(1, t(covar))

    Y_resid <- .lm.fit(x = covar_design,
                       y = t(as.matrix(expr[, -c(1:4)])))$residuals
    X_raw   <- filter_X(geno_mat[samples, , drop = FALSE],
                        missing_rate_thresh = missing.cut,
                        maf_thresh          = maf.cut)
    X_resid <- .lm.fit(x = covar_design, y = X_raw)$residuals

    # marginal sumstats
    target_data <- calculate_sumstat(X_resid, Y_resid) %>%
      separate(SNP, into = c("chrom", "pos", "A2", "A1"), sep = ":", remove = FALSE) %>%
      mutate(chrom = str_remove(chrom, "chr"),
             chrom = as.numeric(chrom),
             pos   = as.numeric(pos))

    # harmonize sketch columns to sumstat alleles
    harm <- sketch_harmonize(target_data  = target_data,
                             ref_variants = sketch_ref %>% select(-variant_id),
                             sketch       = res_sketch$LD_matrix)
    if (ncol(harm$sketch) == 0) {
      message(sprintf("  [%s] No variants survive harmonization in %s - skipping",
                      gene.name, tissue))
      next
    }

    # align individual X to harmonized variant set
    geno_idx <- match(colnames(harm$sketch), colnames(X_resid))
    keep     <- !is.na(geno_idx)
    if (!all(keep)) {
      harm$sketch  <- harm$sketch[, keep, drop = FALSE]
      harm$sumstat <- harm$sumstat[keep, ]
      geno_idx     <- geno_idx[keep]
    }
    if (length(geno_idx) == 0) next
    X_update <- X_resid[, geno_idx, drop = FALSE]
    n        <- nrow(X_update)

    # kriging QC on sketch-derived LD
    sketch_LD      <- crossprod(harm$sketch) / (nrow(harm$sketch) - 1)
    krig           <- kriging_fix(z = harm$sumstat$z, LD = sketch_LD, n = n)
    harm$sumstat$z <- krig$z

    # ---- 5. fit models ----
    susie_res <- susie_indiv(X = X_update, y = Y_resid)   # individual-data (haochen build)

    insample_sumstat <- calculate_sumstat(X_update, Y_resid)
    insample_z  <- insample_sumstat$z
    Xs          <- scale(X_update, center = TRUE, scale = TRUE)
    insample_LD <- crossprod(Xs) / (nrow(Xs) - 1)

    # ---- in-sample panel (haochen build) ----
    insample_res <- safe_susie(
      susie_hc, "insample_res",
      z = insample_z, R = insample_LD, n = n
    )
    
    insample_eb_mix_res <- safe_susie(
      susie_hc, "insample_eb_mix_res",
      z = insample_z, R = insample_LD, n = n,
      R_finite = B_insample, R_mismatch = "eb_mix"
    )
    
    # ---- sketch / KG panel ----
    susie_rss_res <- safe_susie(
      susie_hc, "susie_rss_res",
      z = harm$sumstat$z, X = harm$sketch, n = n
    )
    
    susie_eb_mix_res <- safe_susie(
      susie_hc, "susie_eb_mix_res",
      z = harm$sumstat$z, X = harm$sketch, n = n,
      R_finite = B_sketch, R_mismatch = "eb_mix"
    )
    
    susie_eb_mix_res_new <- safe_susie(
      susie_xw, "susie_eb_mix_res_new",
      z = harm$sumstat$z, X = harm$sketch, n = n,
      R_finite = B_sketch, R_mismatch = "eb_mix",
      check_prior = FALSE
    )
    
    susie_ser_res <- safe_susie(
      susie_hc, "susie_ser_res",
      z = harm$sumstat$z, X = harm$sketch, n = n,
      L = 1, max_iter = 1
    )
    
    # ---- max inter-CS correlation (NA when < 2 CS) ----
    max_cs_corr <- c(
      susie_res           = cs_maxcor(susie_res,            insample_LD),
      insample_res        = cs_maxcor(insample_res,         insample_LD),
      insample_eb_mix_res = cs_maxcor(insample_eb_mix_res,  insample_LD),
      susie_rss_res       = cs_maxcor(susie_rss_res,        sketch_LD),
      susie_eb_mix_res    = cs_maxcor(susie_eb_mix_res,     sketch_LD),
      susie_eb_mix_res_new = cs_maxcor(susie_eb_mix_res_new, sketch_LD),
      susie_ser_res       = cs_maxcor(susie_ser_res,        sketch_LD)
    )
    
    res_susie[[tissue]] <- list(
      variant_ids          = colnames(harm$sketch),
      # --- fits ---
      susie_res            = susie_res,
      insample_res         = insample_res,
      insample_eb_mix_res  = insample_eb_mix_res,
      susie_rss_res        = susie_rss_res,
      susie_eb_mix_res     = susie_eb_mix_res,
      susie_eb_mix_res_new = susie_eb_mix_res_new,
      susie_ser_res        = susie_ser_res,
      # --- diagnostics / bookkeeping ---
      max_cs_corr          = max_cs_corr,
      sample_size          = n,
      flipped_prop         = harm$flipped_prop,
      krig_outliers        = krig$outlier_number
    )
  }

  # ---- 6. save per-gene ----
  if (length(res_susie) == 0) {
    saveRDS("no tissues processed",
            file.path(out_dir, paste0("nores_", gene.name, ".rds")))
  } else {
    saveRDS(res_susie, file.path(out_dir, paste0("res_", gene.name, ".rds")))
  }
}

message(sprintf("[IDX=%d] done.", IDX))

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/code/Gtex_fine.R


In [2]:
%%writefile /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/code/submit_Gtex_fine.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J "Gtex_fine[1-100]"
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 72:00
#BSUB -o /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/log/Gtex_fine.%J.%I.out
#BSUB -e /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/log/Gtex_fine.%J.%I.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

workdir=/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/
mkdir -p ${workdir}/log
code_dir="$workdir/code"
tmp_root="$workdir/tmp"

tmp_dir="$tmp_root/${LSB_JOBID}_${LSB_JOBINDEX}"
mkdir -p "$tmp_dir"

export TMPDIR="$tmp_dir"
export TMP="$tmp_dir"
export TEMP="$tmp_dir"

Rscript "$code_dir/Gtex_fine.R" "$LSB_JOBINDEX"

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/code/submit_Gtex_fine.sh


In [ ]:
[coloc]
parameter: in_dir = path
# directory containing res_*.rds files (per-gene SuSiE results)
parameter: out_dir = path("output")
# directory to write per-gene coloc_*.rds files
parameter: job_size = 50
parameter: walltime = "24h"
parameter: mem = "20G"
parameter: numThreads = 1
parameter: pp_thres = 0.8
# PP.H4 threshold for declaring colocalization / replication
parameter: container = ""

# discover one input file per gene and produce one output file per gene
input: [str(f) for f in path(in_dir).glob("res_*.rds")], group_by = 1
output: f'{out_dir:a}/coloc_{_input:bn}.rds'

task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'

R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    suppressPackageStartupMessages({
      library(tidyverse)
      library(pecotmr)
      library(coloc)
    })

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    in_file  <- ${_input:r}
    out_file <- ${_output:r}
    gene     <- str_remove(basename(in_file), "^res_") %>% str_remove("\\.rds$")
    pp_thres <- ${pp_thres}
    cat(sprintf("[%s] gene: %s\n", format(Sys.time(), "%H:%M:%S"), gene))
    flush.console()



    sumstat_methods <- c(
        "susie_res",
        "insample_res",
      "insample_eb_mix_res",
      "susie_rss_res",
      "susie_eb_mix_res",
      "susie_eb_mix_res_new",
      "susie_ser_res"
    )

    # ---- diagnostic extraction --------------------------------------------

    flag_or_na <- function(x) if (is.null(x) || length(x) == 0) NA else as.logical(x)[1]

    get_diag <- function(fit) {
      diag <- fit$R_finite_diagnostics       # NULL for non-EB fits -> all NA
      list(
        B_corrected        = as.numeric(fit$B_corrected %||% NA_real_),
        lambda_bias        = as.numeric(fit$lambda_bias %||% NA_real_),
        Q_art              = as.numeric(diag$Q_art %||% NA_real_),
        D_CS_max           = if (!is.null(diag$bf_attenuation$cs_max))
                               suppressWarnings(max(diag$bf_attenuation$cs_max, na.rm = TRUE))
                             else NA_real_,
        max_abs_z          = if (!is.null(diag$ser_model$z))
                               suppressWarnings(max(abs(diag$ser_model$z), na.rm = TRUE))
                             else NA_real_,
        artifact_evaluable = flag_or_na(diag$artifact_evaluable),
        reliability_flag   = flag_or_na(diag$R_reliability_flag),
        sensitivity_flag   = flag_or_na(diag$R_sensitivity_flag),
        artifact_flag      = flag_or_na(fit$artifact_flag),
        prior_flag         = flag_or_na(fit$prior_variance_flag)
      )
    }

    coloc_pair <- function(insample, outsample) {
      colnames(insample$alpha)        <- colnames(outsample$alpha)
      colnames(insample$mu)           <- colnames(outsample$mu)
      colnames(insample$mu2)          <- colnames(outsample$mu2)
      colnames(insample$lbf_variable) <- colnames(outsample$lbf_variable)
      names(insample$X_column_scale_factors) <- names(outsample$X_column_scale_factors)
      names(insample$pip)             <- names(outsample$pip)
      coloc.susie(insample, outsample)
    }

    one_record <- function(insample, outsample, gene, tissue, method, max_corr = NA_real_) {
      n_in  <- length(insample$sets$cs)
      n_out <- length(outsample$sets$cs)
      if (n_in == 0 && n_out == 0) return(NULL)

      dg <- get_diag(outsample)             # metrics describe the tested (outsample) fit

      rec <- tibble(
        gene = gene, tissue = tissue, method = method,
        n_insample_cs = n_in, n_outsample_cs = n_out,
        replicated_insample = NA_integer_, replicated_outsample = NA_integer_,
        # ---- requested metrics ----
        B_corrected = dg$B_corrected,
        max_corr         = as.numeric(max_corr),
        lambda_pop       = dg$lambda_bias,
        Q_art            = dg$Q_art,
        DCS_max          = dg$D_CS_max,
        artifact_flag    = dg$artifact_flag,
        sensitivity_flag = dg$sensitivity_flag,
        reliability_flag = dg$reliability_flag,
        prior_flag = dg$prior_flag,
        max_abs_z = dg$max_abs_z,
        error = NA_character_
      )
      if (n_in  == 0) { rec$replicated_outsample <- 0L; return(rec) }
      if (n_out == 0) { rec$replicated_insample  <- 0L; return(rec) }
        tryCatch({
        cr <- coloc_pair(insample, outsample)
        sm <- cr$summary %>%
          mutate(coloc = PP.H4.abf > pmax(PP.H0.abf, PP.H1.abf, PP.H2.abf, PP.H3.abf))
        s1 <- sm %>% group_by(hit1) %>% summarize(coloc = any(coloc, na.rm = TRUE), .groups = "drop")
        s2 <- sm %>% group_by(hit2) %>% summarize(coloc = any(coloc, na.rm = TRUE), .groups = "drop")
        rec$replicated_insample  <- sum(s1$coloc)
        rec$replicated_outsample <- sum(s2$coloc)
        rec
      }, error = function(e) { rec$error <- conditionMessage(e); rec })
    }

    # ---- run ----
    res <- tryCatch(readRDS(in_file), error = function(e) NULL)
    if (is.null(res) || is.character(res) || !is.list(res) || length(res) == 0) {
      saveRDS(NULL, out_file)
      cat(sprintf("[%s] empty input, wrote NULL\n", gene))
      quit(status = 0)
    }

    recs <- map_dfr(names(res), function(tissue) {
      cat(sprintf("  [%s] tissue: %s\n", gene, tissue))
      flush.console()
      result   <- res[[tissue]]
      insample <- result$susie_res
      if (is.null(insample)) return(NULL)

      mcc <- result$max_cs_corr            # named vector by method; NULL if not stored

      base_recs <- map_dfr(sumstat_methods, function(method) {
        outsample <- result[[method]]
        if (is.null(outsample)) return(NULL)
        one_record(insample, outsample, gene, tissue, method,
                   max_corr = mcc[[method]] %||% NA_real_)
      })

      # hybrid: SER if R is unreliable, else eb
      eb   <- result$susie_eb_res
      ser  <- result$susie_ser_res
      flag <- eb$R_finite_diagnostics$R_reliability_flag
      hybrid_rec <- NULL
      if (!is.null(eb) && !is.null(ser) && !is.null(flag) && !is.na(flag)) {
        chosen     <- if (isTRUE(flag)) ser else eb
        chosen_nm  <- if (isTRUE(flag)) "susie_ser_res" else "susie_eb_mix_res"
        hybrid_rec <- one_record(insample, chosen, gene, tissue, "eb_or_ser",
                                 max_corr = mcc[[chosen_nm]] %||% NA_real_)
      }

      bind_rows(base_recs, hybrid_rec)
    })

    saveRDS(recs, out_file)
    cat(sprintf("[%s] saved %d records\n", gene, nrow(recs %||% tibble())))

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/"
job_name="coloc"
job_dir="${base_dir}/${job_name}"

# Source: where the per-gene SuSiE result RDS files live
src_dir="${base_dir}/res"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=25000]
#BSUB -W 36:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/16.Fig5A_5C.GTEx_Coloc.ipynb" coloc \
    --in-dir SRC_DIR \
    --out-dir WORK_DIR/result/ \
    --mem 40G -J 50
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

output_script="${job_name}.sh.submit"
sed -e "s|WORK_DIR|${work_dir}|g" \
    -e "s|SRC_DIR|${src_dir}|g" \
    -e "s|JOB_NAME|${job_name}|g" \
    "${base_sh}" > "${output_script}"
 sb "${output_script}"